# LARA Stack Lab auf Colab: L4 und A100

Ein Notebook für beide GPUs. **Laufzeit → Laufzeittyp ändern → GPU** wählen; zunächst L4, für den Vergleich später A100. Die tatsächlich zugeteilte Karte wird erfasst. Projektmodell, Roboter-API und unabhängige Physikprüfung bleiben dieselben.

Zellen der Reihe nach ausführen. Die Einrichtung installiert eine isolierte Python-Umgebung und lädt nur die ausgewählten Modellgewichte. Die GPU-Prüfung verlangt einen CUDA-Backend und echte Modell-Offloading-Meldungen; ein stiller Wechsel auf CPU findet nicht statt.

Dieses Notebook führt Entwicklungstests aus. Eine Modell-Fertigmeldung ist kein physischer Stapelerfolg. Es wurde hier noch keine L4-/A100-Leistung vorab eingetragen.

In [ ]:
MODELS = ["granite8b"]       # z.B. ["qwen9b"], ["lfm2.6b"] oder mehrere Aliasse
CASES = ["move-right", "red-on-blue"]
BLOCKS = [1, 4]
SCENE_SEEDS = [42]
GENERATION_SEEDS = [109]
MAX_TOKENS = 2048
THINKING_BUDGET = 768
THREADS = 2                  # Für L4/A100-Vergleich gleich lassen
CUDA_RUNTIME = "auto"        # "source" erzwingt denselben Quellbuild auf beiden Karten
SAVE_TO_DRIVE = True         # Einmal Google-Drive-Zugriff erlauben
SHOW_VIDEO = True            # Separater Bewegungs-Vorversuch als Video
COMPUTE_UNITS_PER_HOUR = None  # Optional: aktuellen Wert aus der Colab-Anzeige eintragen
REPO_REF = "3e7cc43632bb6f51f0f079223ae018799bf64f7b"  # Derselbe Code für L4 und A100

In [ ]:
import os, sys, json, subprocess, signal, time, re, csv, io
from pathlib import Path
from datetime import datetime, timezone

def command(args, *, allowed=(0,), cwd=None, env=None):
    print("$", " ".join(map(str, args)), flush=True)
    process = subprocess.Popen(list(map(str, args)), cwd=cwd, env=env, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1, start_new_session=True)
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        code = process.wait()
    except KeyboardInterrupt:
        os.killpg(process.pid, signal.SIGINT)
        try:
            process.wait(timeout=20)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGKILL)
            process.wait()
        raise
    finally:
        process.stdout.close()
    if code not in allowed:
        raise RuntimeError(f"Befehl fehlgeschlagen, Exit-Code {code}; Ausgabe oben prüfen.")
    return code

gpu_text = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                                    "--format=csv,noheader,nounits"], text=True)
print(gpu_text)
gpu_name = next(csv.reader(io.StringIO(gpu_text)))[0].strip()
gpu_tag = re.sub(r"[^A-Za-z0-9_-]+", "_", gpu_name)
if COMPUTE_UNITS_PER_HOUR is not None:
    import math
    assert isinstance(COMPUTE_UNITS_PER_HOUR, (int,float)) and math.isfinite(COMPUTE_UNITS_PER_HOUR) and COMPUTE_UNITS_PER_HOUR > 0

## Projekt einrichten

Der Projektcode läuft in `.venv`, damit sich seine Pakete nicht mit dem Notebook-Kernel vermischen. Falls der Kernel nicht Python 3.12/3.13 nutzt, wird Python 3.12 mit `uv` eingerichtet. Ein vorhandener Checkout wird wiederverwendet, statt während laufender Versuche aktualisiert zu werden. Der voreingestellte `REPO_REF` hält denselben Code für L4 und A100 fest. Für den Vergleich diesen Wert auf beiden Karten beibehalten; die tatsächlich verwendete Commit-SHA wird zusätzlich ausgegeben.

In [ ]:
PROJECT = Path("/content/lara-stack-lab")
if not PROJECT.exists():
    command(["git", "clone", "https://github.com/carlo2004/Simulation-Neura-LARA-5.git", PROJECT])
    if REPO_REF != "main":
        command(["git", "fetch", "--depth", "1", "origin", REPO_REF], cwd=PROJECT)
        command(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=PROJECT)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT, text=True).strip()
if REPO_REF != "main" and not commit.startswith(REPO_REF):
    raise RuntimeError("Vorhandener Checkout gehört zu einem anderen Commit. Neue Colab-Laufzeit verwenden.")
print("Für den GPU-Vergleich festhalten: REPO_REF =", repr(commit))
bootstrap_python = sys.executable
if sys.version_info[:2] not in ((3,12), (3,13)):
    command([sys.executable, "-m", "pip", "install", "--quiet", "uv"])
    command([sys.executable, "-m", "uv", "python", "install", "3.12"])
    bootstrap_python = subprocess.check_output([sys.executable, "-m", "uv", "python", "find", "3.12"], text=True).strip()
command([bootstrap_python, "bootstrap.py"], cwd=PROJECT)
PYTHON = PROJECT/".venv/bin/python"
ENV = dict(os.environ, OPENBLAS_NUM_THREADS="1", MUJOCO_GL="osmesa", PYOPENGL_PLATFORM="osmesa",
           STACKLAB_CUDA_RUNTIME=CUDA_RUNTIME)
def lab(*args, allowed=(0,)):
    return command([PYTHON, "-m", "stacklab", *args], cwd=PROJECT, env=ENV, allowed=allowed)
lab("--version")
lab("models")
if SHOW_VIDEO:
    import ctypes.util
    if not ctypes.util.find_library("OSMesa"):
        command(["apt-get", "update", "-qq"])
        command(["apt-get", "install", "-y", "-qq", "libosmesa6"])

In [ ]:
BACKUP_DIR = Path("/content/lara-backups")
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BACKUP_DIR = Path("/content/drive/MyDrive/LARA-Stack-Lab/experiments")
BACKUP_DIR.mkdir(parents=True, exist_ok=True)
print("Sicherungen:", BACKUP_DIR)

## CUDA und Physik prüfen

Für kompatible Systeme werden geprüfte CUDA-Binärpakete derselben llama.cpp-Version wie lokal verwendet. Auf älteren Colab-Systemen mit GLIBC vor 2.38 baut `auto` denselben festgelegten Quellstand selbst. **Dory** verwendet immer seinen eigenen Quellbuild. Quellbuilds benötigen zusätzlich CMake, C++-Compiler und `nvcc`; der erste Build dauert länger und zeigt seinen Fortschritt an. Es wird nur das ausgewählte Modell heruntergeladen. Ein GPU-/Treiberfehler stoppt die Einrichtung vor dem Gewichtsdownload.

Der folgende `demo`-Aufruf ist ein **handgeschriebenes Kontrollprogramm**, kein Erfolg des Sprachmodells. Er prüft, dass die Simulation selbst stapeln kann.

In [ ]:
import shutil
recipe_code = "import json,sys; from stacklab.llm.catalog import lock; print(json.dumps([dict(model=m,kind=lock(m,device='cuda').get('cuda_runtime_kind')) for m in sys.argv[1:]]))"
recipes = json.loads(subprocess.check_output([PYTHON, "-c", recipe_code, *MODELS], cwd=PROJECT, env=ENV, text=True))
print("Gewählte CUDA-Runtimes:", recipes)
if any(r["kind"] == "source" for r in recipes):
    if not (PYTHON.parent/"cmake").is_file() and not shutil.which("cmake"):
        command([PYTHON,"-m","pip","install","cmake"],cwd=PROJECT,env=ENV)
    if not shutil.which("g++") or not shutil.which("make"):
        command(["apt-get","update","-qq"])
        command(["apt-get","install","-y","-qq","g++","make"])
    if not shutil.which("nvcc",path=ENV["PATH"]):
        if Path("/usr/local/cuda/bin/nvcc").is_file():
            ENV["PATH"] = "/usr/local/cuda/bin:"+ENV["PATH"]
        else:
            raise RuntimeError("CUDA Toolkit (nvcc) fehlt. Eine aktuelle Colab-GPU-Laufzeit verwenden.")
for model in MODELS:
    lab("setup", "--model", model, "--device", "cuda")
    lab("doctor", "--model", model, "--device", "cuda")
control_dir = Path("/content/lara-control")/datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
lab("demo", "--output", control_dir)

## Optional ansehen, was das Modell ausführt

Ein separater Bewegungs-Vorversuch erzeugt ein Video der **tatsächlichen** simulierten Ausführung. Code, Rückmeldungen und gemessene Tokens/s stehen im Protokoll. Rendering nutzt OSMesa auf der CPU und funktioniert unabhängig von den Grafikfähigkeiten der L4/A100. Die folgenden Hauptversuche zeichnen standardmäßig kein Video auf, um Rendering-Aufwand aus dem Vergleich herauszuhalten.

In [ ]:
if SHOW_VIDEO:
    from IPython.display import Video, display
    preview = Path("/content/lara-preview")/(gpu_tag+"_"+datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f"))
    lab("benchmark", "--model", MODELS[0], "--device", "cuda", "--case", "move-right",
        "--feedback-rounds", "1", "--threads", str(THREADS), "--max-tokens", str(MAX_TOKENS),
        "--thinking-budget", str(THINKING_BUDGET), "--video", "--output", preview, allowed=(0,1))
    video = preview/"move-right/simulation.mp4"
    result = json.loads((preview/"move-right/result.json").read_text())
    turns = result.get("execution", {}).get("rounds", [])
    if not any((t.get("generation", {}).get("hardware") or {}).get("offloaded_layers", 0) > 0 for t in turns):
        raise RuntimeError("Der Modellstart hat keine GPU-Ausführung bestätigt. Vorschau-Protokoll unter "+str(preview)+" prüfen.")
    print("Physischer Erfolg des Vorversuchs:", result["success"])
    if video.is_file():
        display(Video(str(video), embed=True))

## Gleiche Aufgaben mit einem oder mehreren Programmblöcken

Es entsteht ein neuer eingefrorener Entwicklungsplan. Jede Episode hat eine frische Szene und einen neuen Modellprozess; der Betriebssystem-Dateicache wird nicht geleert. Nach jeder Episode wird eine ZIP-Sicherung erstellt. Strg+C beendet den laufenden Worker und versucht, auch dessen Teilprotokoll zu sichern. Ein harter Sitzungsabbruch kann die zuletzt laufende Episode verlieren; die vorherigen Sicherungen bleiben erhalten.

Der Exit-Code `1` gehört zu dokumentierten Modellfehlschlägen. Er ist kein Grund, Ergebnisse zu entfernen oder Programme nachträglich zu reparieren. Die optionale CU-Schätzung nutzt den von dir eingetragenen Stundensatz und die Laufzeit dieser Versuchszelle einschließlich Sicherungen, **ohne vorherige Einrichtung und Downloads**. Es ist keine Google-Abrechnung.

In [ ]:
RESULT_DIR = Path("/content/lara-results")/(gpu_tag+"_"+datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f"))
lab("experiment", "plan", "--output", RESULT_DIR, "--models", *MODELS,
    "--device", "cuda", "--cases", *CASES, "--blocks", *map(str,BLOCKS),
    "--scene-seeds", *map(str,SCENE_SEEDS), "--generation-seeds", *map(str,GENERATION_SEEDS),
    "--threads", str(THREADS), "--max-tokens", str(MAX_TOKENS), "--thinking-budget", str(THINKING_BUDGET))
session = dict(gpu_report=gpu_text, software_commit=commit, units_per_hour_hint=COMPUTE_UNITS_PER_HOUR,
               note="CU estimate only; excludes setup/download/preview and is not provider billing")
(RESULT_DIR/"colab-session.json").write_text(json.dumps(session, indent=2)+"\n")
started = time.monotonic()
try:
    lab("experiment", "run", RESULT_DIR, "--backup-dir", BACKUP_DIR, allowed=(0,1))
finally:
    session["experiment_cell_seconds"] = time.monotonic()-started
    session["estimated_compute_units"] = (session["experiment_cell_seconds"]/3600*COMPUTE_UNITS_PER_HOUR
                                          if COMPUTE_UNITS_PER_HOUR is not None else None)
    (RESULT_DIR/"colab-session.json").write_text(json.dumps(session, indent=2)+"\n")
    lab("experiment", "backup", RESULT_DIR, "--output", BACKUP_DIR)
print(json.dumps(session, indent=2))
print("Ergebnisse:", RESULT_DIR)

In [ ]:
from IPython.display import Markdown, display
display(Markdown((RESULT_DIR/"report.md").read_text()))

## L4 und A100 vergleichen

Für die A100 eine neue GPU-Laufzeit auswählen und dasselbe Notebook erneut ausführen. Den voreingestellten `REPO_REF` beibehalten; Modell, Fälle, Seeds, Thinking-Budget und Threads unverändert lassen. Dieselbe Drive-Sicherung verwenden. Die folgende Tabelle führt **alle** gefundenen Entwicklungspläne auf, einschließlich Fehlschlägen und unvollständigen Plänen. Sie kombiniert keine unterschiedlichen Protokolle zu einer gemeinsamen Erfolgsquote. Nur Zeilen mit passenden Fall-/Bedingungs-IDs und Software-/Promptbedingungen direkt vergleichen.

In [ ]:
import zipfile, hashlib, html
from IPython.display import HTML, display
rows=[]
budgets=[]
for archive_path in sorted(BACKUP_DIR.glob("*.zip")):
    try:
        with zipfile.ZipFile(archive_path) as archive:
            manifest=json.loads(archive.read("manifest.json"))
            summary=json.loads(archive.read("summary.json"))
            content={k:v for k,v in manifest.items() if k!="plan_sha256"}
            canonical=json.dumps(content,sort_keys=True,ensure_ascii=False,allow_nan=False,separators=(",",":"))
            assert hashlib.sha256(canonical.encode()).hexdigest()==manifest["plan_sha256"]
            assert summary["plan_sha256"]==manifest["plan_sha256"]
            gpu=", ".join(g["name"] for g in manifest["environment"].get("gpus",[])) or "CPU/unknown"
            for group in summary["groups"]:
                episodes=[e for e in summary["episodes"] if e["condition_id"]==group["condition_id"] and e["case"]==group["case"]]
                wall=[e["wall_seconds"] for e in episodes if e.get("wall_seconds") is not None]
                rows.append(dict(plan=manifest["plan_sha256"][:12],gpu=gpu,model=group["model"],blocks=group["blocks"],
                                 case=group["case"],condition_id=group["condition_id"],success=group["success"],
                                 scored=group["scored"],planned=group["planned"],infrastructure=group["infrastructure_error"],
                                 open=group["pending"]+group["incomplete"],cancelled=group["cancelled"],
                                 tokens_per_second=group["performance"].get("decode_tokens_per_second"),
                                 partial=group["performance"].get("partial"),
                                 mean_wall_seconds=sum(wall)/len(wall) if wall else None))
            if "colab-session.json" in archive.namelist():
                budget=json.loads(archive.read("colab-session.json"))
                budgets.append(dict(plan=manifest["plan_sha256"][:12],gpu=gpu,
                                    units_per_hour=budget.get("units_per_hour_hint"),
                                    estimated_units=budget.get("estimated_compute_units")))
    except Exception as error:
        print("Archiv konnte nicht ausgewertet werden:",archive_path.name,type(error).__name__,str(error))
if rows:
    fields=list(rows[0])
    table="<table><tr>"+"".join("<th>"+html.escape(k)+"</th>" for k in fields)+"</tr>"
    for row in rows:
        table+="<tr>"+"".join("<td>"+html.escape(str(row[k]))+"</td>" for k in fields)+"</tr>"
    display(HTML(table+"</table>"))
    with (BACKUP_DIR/"comparison.csv").open("w",newline="",encoding="utf-8") as stream:
        writer=csv.DictWriter(stream,fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
    print("Vergleichstabelle:",BACKUP_DIR/"comparison.csv")
print("Optionale Budget-Schätzungen (keine Abrechnung):",json.dumps(budgets,indent=2))

## Weiterarbeiten

Im Notebook kannst du über `lab("chat", "--headless", "--device", "cuda", "--model", MODELS[0], "--prompt", "Bewege den Greifer 5 cm nach rechts.")` einen freien Prompt testen. Jeder neue Chat-Aufruf startet eine neue Szene. Freie Sprache ohne extern vorgegebenes Ziel erhält keine behauptete unabhängige Erfolgsbewertung.

Sicherungen enthalten Rohantworten, tatsächlichen Code, Rückmeldungen, Physikzustände und Tokens/s. Vor dem Beenden prüfen, dass die ZIP-Dateien in Drive angekommen sind. Ohne Drive sind die Daten nur in der aktuellen Colab-VM; dann die ZIP-Datei über den Dateibrowser herunterladen. Nach der Nutzung die Laufzeit trennen und löschen.

[Projekt](https://github.com/carlo2004/Simulation-Neura-LARA-5) · [Colab-Anleitung](https://github.com/carlo2004/Simulation-Neura-LARA-5/blob/main/docs/colab.md)